In [1]:
from physics_equations import *

In [2]:
def solve_entrained_parcel_state_debug(theta_target, q_t_total, p, p_ref, theta_type, T_initial_guess, 
                                       max_iter=50, tolerance=0.01, debug=True):
    """
    Debug version with detailed logging and improved convergence
    """
    
    T_guess = T_initial_guess
    
    if debug:
        print(f"\n=== Solving for theta_target={theta_target:.2f} K, q_t={q_t_total*1000:.3f} g/kg, p={p/100:.1f} hPa ===")
    
    # Store previous values for secant method
    T_prev = None
    theta_prev = None
    
    for iteration in range(max_iter):
        # Calculate saturation mixing ratio at current temperature
        try:
            w_sat_val = w_sat(T_guess, p)
        except:
            print(f"⚠️ Error calculating w_sat at T={T_guess:.2f} K, p={p:.0f} Pa")
            return T_guess, 0.0, 0.0
        
        # Partition total water between vapor and liquid
        if q_t_total > w_sat_val:
            # Saturated - some condensation has occurred
            q_v = w_sat_val  # vapor is at saturation
            q_l = q_t_total - w_sat_val  # excess becomes liquid
            saturation_status = "SATURATED"
        else:
            # Unsaturated - all water is vapor
            q_v = q_t_total
            q_l = 0.0
            saturation_status = "UNSATURATED"
        
        # Calculate potential temperature with current T and water partitioning
        try:
            theta_calc = theta_entrain(T_guess, p, p_ref, q_v, q_t_total, theta_type)
        except Exception as e:
            print(f"⚠️ Error calculating theta_entrain: {e}")
            print(f"   T={T_guess:.2f}, p={p:.0f}, q_v={q_v*1000:.3f}, q_t={q_t_total*1000:.3f}")
            return T_guess, q_v, q_l
        
        # Check for numerical issues
        if not np.isfinite(theta_calc):
            print(f"⚠️ Non-finite theta_calc = {theta_calc} at T={T_guess:.2f} K")
            return T_guess, q_v, q_l
        
        # Calculate error
        error = theta_calc - theta_target
        
        if debug and (iteration < 5 or iteration % 10 == 0):
            print(f"  Iter {iteration:2d}: T={T_guess:6.2f} K, theta={theta_calc:6.2f} K, error={error:6.3f} K, {saturation_status}")
            print(f"           q_v={q_v*1000:5.2f} g/kg, q_l={q_l*1000:5.2f} g/kg, w_sat={w_sat_val*1000:5.2f} g/kg")
        
        # Check convergence
        if abs(error) < tolerance:
            if debug:
                print(f"  ✅ CONVERGED in {iteration+1} iterations")
            return T_guess, q_v, q_l
        
        # Calculate temperature adjustment
        if iteration == 0:
            # First iteration - simple proportional adjustment
            # For most theta types, dtheta/dT is roughly theta/T
            dT = -error * T_guess / max(theta_calc, 100)  # avoid division by small numbers
            dT = np.clip(dT, -10.0, 10.0)  # limit first step
            
        elif T_prev is not None and theta_prev is not None:
            # Use secant method with previous point
            if abs(theta_calc - theta_prev) > 1e-10:
                dT_secant = (T_guess - T_prev) * error / (theta_calc - theta_prev)
                dT = -dT_secant
            else:
                # Fall back to finite difference
                dT = -error * 0.1
        else:
            # Use finite difference approximation
            dT_test = min(1.0, 0.01 * T_guess)  # 1% perturbation or 1K, whichever is smaller
            T_test = T_guess + dT_test
            
            try:
                w_sat_test = w_sat(T_test, p)
                if q_t_total > w_sat_test:
                    q_v_test = w_sat_test
                else:
                    q_v_test = q_t_total
                
                theta_test = theta_entrain(T_test, p, p_ref, q_v_test, q_t_total, theta_type)
                
                if abs(theta_test - theta_calc) > 1e-10:
                    d_theta_dT = (theta_test - theta_calc) / dT_test
                    dT = -error / d_theta_dT
                else:
                    dT = -error * 0.1
            except:
                dT = -error * 0.1
        
        # Store values for next iteration
        T_prev = T_guess
        theta_prev = theta_calc
        
        # Apply temperature adjustment with adaptive damping
        # Reduce damping as we get closer to solution
        damping = max(0.3, min(0.8, 1.0 / (1 + iteration/10)))
        dT = np.clip(dT, -5.0, 5.0)  # limit step size
        T_guess = T_guess + damping * dT
        
        # Keep temperature in reasonable bounds
        T_guess = np.clip(T_guess, 150.0, 400.0)
        
        # Check if we're oscillating
        if iteration > 10 and abs(error) > abs(theta_prev - theta_target):
            # Error is increasing - reduce step size
            T_guess = 0.5 * (T_guess + T_prev)
            if debug:
                print(f"  ⚠️ Error increasing, averaging with previous T")
    
    # If we reach here, convergence failed
    print(f"⚠️ Warning: solve_entrained_parcel_state did not converge after {max_iter} iterations")
    print(f"   Final: T={T_guess:.2f} K, theta={theta_calc:.2f} K, target={theta_target:.2f} K")
    print(f"   Final error: {error:.6f} K, tolerance: {tolerance:.6f} K")
    print(f"   q_t={q_t_total*1000:.3f} g/kg, p={p/100:.1f} hPa, theta_type={theta_type}")
    
    # Return best estimate
    w_sat_val = w_sat(T_guess, p)
    if q_t_total > w_sat_val:
        q_v = w_sat_val
        q_l = q_t_total - w_sat_val
    else:
        q_v = q_t_total
        q_l = 0.0
        
    return T_guess, q_v, q_l


def solve_entrained_parcel_state_robust(theta_target, q_t_total, p, p_ref, theta_type, T_initial_guess):
    """
    More robust solver using scipy.optimize as backup
    """
    
    # First try the custom iterative method
    try:
        T_sol, q_v_sol, q_l_sol = solve_entrained_parcel_state_debug(
            theta_target, q_t_total, p, p_ref, theta_type, T_initial_guess, 
            max_iter=30, tolerance=0.01, debug=False
        )
        
        # Verify the solution
        w_sat_val = w_sat(T_sol, p)
        q_v_check = min(q_t_total, w_sat_val)
        theta_check = theta_entrain(T_sol, p, p_ref, q_v_check, q_t_total, theta_type)
        
        if abs(theta_check - theta_target) < 0.1:  # Good enough
            return T_sol, q_v_sol, q_l_sol
    except:
        pass
    
    print(f"⚠️ Falling back to scipy solver...")
    
    # Fallback to scipy
    from scipy.optimize import brentq, minimize_scalar
    
    def theta_error(T):
        T = max(150.0, min(400.0, T))  # bounds
        try:
            w_sat_val = w_sat(T, p)
            q_v = min(q_t_total, w_sat_val)
            theta_calc = theta_entrain(T, p, p_ref, q_v, q_t_total, theta_type)
            return theta_calc - theta_target
        except:
            return 1e6  # large error if calculation fails
    
    try:
        # Try Brent's method if we can find a bracketing interval
        T_low, T_high = T_initial_guess - 50, T_initial_guess + 50
        T_low = max(150.0, T_low)
        T_high = min(400.0, T_high)
        
        error_low = theta_error(T_low)
        error_high = theta_error(T_high)
        
        if error_low * error_high < 0:  # Different signs - can use brentq
            T_solution = brentq(theta_error, T_low, T_high, xtol=0.01)
        else:
            # Use minimize_scalar
            result = minimize_scalar(lambda T: abs(theta_error(T)), 
                                   bounds=(150.0, 400.0), method='bounded')
            T_solution = result.x
            
        # Calculate final water partitioning
        w_sat_val = w_sat(T_solution, p)
        if q_t_total > w_sat_val:
            q_v = w_sat_val
            q_l = q_t_total - w_sat_val
        else:
            q_v = q_t_total
            q_l = 0.0
            
        return T_solution, q_v, q_l
        
    except Exception as e:
        print(f"⚠️ Scipy solver also failed: {e}")
        # Last resort - return initial guess with reasonable partitioning
        w_sat_val = w_sat(T_initial_guess, p)
        if q_t_total > w_sat_val:
            q_v = w_sat_val
            q_l = q_t_total - w_sat_val
        else:
            q_v = q_t_total
            q_l = 0.0
        return T_initial_guess, q_v, q_l


# Simple test function
def test_solver():
    """Test the solver with known values"""
    print("=== Testing Solver ===")
    
    # Test case: should be easy to solve
    T_test = 280.0  # K
    p_test = 85000.0  # Pa
    p_ref = 100000.0  # Pa
    q_t_test = 0.008  # kg/kg
    theta_type = 'theta_e'
    
    # Calculate target theta
    w_sat_val = w_sat(T_test, p_test)
    q_v_test = min(q_t_test, w_sat_val)
    theta_target = theta_entrain(T_test, p_test, p_ref, q_v_test, q_t_test, theta_type)
    
    print(f"Target: T={T_test:.2f} K, theta={theta_target:.2f} K")
    
    # Now solve for it
    T_sol, q_v_sol, q_l_sol = solve_entrained_parcel_state_debug(
        theta_target, q_t_test, p_test, p_ref, theta_type, T_test + 10, debug=True
    )
    
    print(f"Solution: T={T_sol:.2f} K (error: {T_sol-T_test:.3f} K)")

In [3]:
test_solver()

=== Testing Solver ===


TypeError: 'float' object is not subscriptable